# 07 — Blocking / candidate generation (Phase 7)

## Concept
**Blocking** = for each Source 1 entity, quickly pick a *short list* of Source 2/3 records that could plausibly be the same
business. Only those **candidates** are compared carefully later.

## Why do we need it?
Comparing every S1 entity with every same-country S2/S3 record would be **6.7 trillion** comparisons on the test set (notebook 04).
Blocking brings that down to ~50 per entity.

## Simple example
```
S1: "Maure Williams Colombier Inc, 85 Wayne Avenue, Ticonderoga, NY"
pool: 3 million US records
blocking: records that share a RARE key with it
   -> "85_wayne" (address word pair, appears in 3 records)
   -> "maurewilliamscolombier" (joined name, appears in 2 records)
   -> ... 50 candidates
```

## The two numbers that describe a blocking method
- **Candidate recall** = share of true pairs that are among the candidates. It is the **ceiling** of the whole system:
  a match that is not a candidate can never be predicted.
- **Candidates per entity** = how much work the model has to do afterwards (and how many look-alikes it must reject).

The organizers also look at the **reduction ratio** = 1 − candidates / all possible pairs.

## Common mistakes
- Blocking on *common* words (`street`, `llc`): millions of candidates, and true matches get crowded out.
- Blocking on *exact* names: only ~11% of true pairs have the same name (notebook 03).
- Keeping too few candidates per entity: look-alikes push the true match out of the list.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.blocking_keys import record_keys
from src.preprocessing import ADDRESS_STEPS, NAME_STEPS, normalize

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 200)
EXP = PROJECT_ROOT / "experiments"

---
## 1. Diagnosis: why did the baseline lose 41% of the matches?

`experiments/phase7_diagnose.py` looked at every true pair of the 20k **tuning** entities (validation untouched)
and computed, for several key types, the document frequency (df) of the **rarest key both records share**.
A pair can only be found by a blocking rule that allows keys at least that common.

In [2]:
diagnosis = pd.read_csv(EXP / "phase7_diagnosis.tsv", sep="\t")
thresholds = [100, 300, 1000, 3000, 10_000]
key_sets = {
    "single words (baseline keys)": ["nw", "aw"],
    "address word pairs": ["ab"],
    "name word pairs": ["nb"],
    "joined name": ["nj"],
    "all 5 key types": ["nw", "aw", "nb", "ab", "nj"],
}
reach = {}
for label, types in key_sets.items():
    rarest = diagnosis[[f"min_df_{t}" for t in types]].min(axis=1)
    reach[label] = [round((rarest <= t).mean() * 100, 1) for t in thresholds]
pd.DataFrame(reach, index=[f"df <= {t:,}" for t in thresholds]).T

,df <= 100,df <= 300,"df <= 1,000","df <= 3,000","df <= 10,000"
single words (baseline keys),44.8,67.1,82.5,92.7,97.7
address word pairs,82.4,86.7,89.9,91.4,92.4
name word pairs,54.4,64.5,66.7,68.7,69.6
joined name,47.0,50.3,52.6,52.6,52.6
all 5 key types,94.3,97.1,98.4,99.0,99.5


**Reading the table**
- With single words and `df <= 1000`, 82.5% of true pairs were *reachable* — yet the baseline found only 58%.
  The missing ~25 points were lost to **ranking**: many look-alike records shared the same words and pushed the true match
  out of the top 50.
- **Address word pairs** are far more precise than single words: `85_wayne` is rare even though `85` and `wayne` are common.
  82% of true pairs share an address pair that appears in ≤ 100 records.
- With **all 5 key types**, 98.4% of true pairs share a key with df ≤ 1000.

---
## 2. The five key types on a real example

All keys carry the country (`US|...`), so different countries can never match — and France works automatically.

| code | key type | example |
|---|---|---|
| `nw` | name word | `US|nw|colombier` |
| `aw` | address word | `US|aw|wayne` |
| `nb` | name word pair | `US|nb|williams_colombier` |
| `ab` | address word pair | `US|ab|85_wayne` |
| `nj` | joined name (legal words removed, no spaces) | `US|nj|maurewilliamscolombier` — matches website names |

In [3]:
cache = PROJECT_ROOT / "cache"
s1_example = pd.read_parquet(cache / "train_source1.parquet", filters=[("entity_id", "=", "S1-965667")])
matches = pd.read_parquet(cache / "train_pool_norm.parquet",
                          filters=[("entity_id", "in", ["S2-681193310", "S2-743505751", "S3-775321672", "S3-11291185", "S3-860443364"])])

s1_name = normalize(s1_example["business_name"], NAME_STEPS)
s1_addr = normalize(s1_example["business_address"], ADDRESS_STEPS)
s1_keys = set(record_keys(s1_name, s1_addr, s1_example["country"])["key"])
print("S1:", s1_name.iloc[0], "|", s1_addr.iloc[0])
print(len(s1_keys), "keys, e.g.", sorted(s1_keys)[:8])

rows = []
for _, m in matches.iterrows():
    m_keys = set(record_keys(pd.Series([m["name_norm"]]), pd.Series([m["addr_norm"]]), pd.Series([m["country"]]))["key"])
    rows.append({"match": m["entity_id"], "name": m["name_norm"], "address": m["addr_norm"],
                 "shared keys": sorted(s1_keys & m_keys)})
pd.DataFrame(rows)

S1: maure williams colombier inc | 85 wayne avenue ticonderoga ny
17 keys, e.g. ['US|ab|85_wayne', 'US|ab|avenue_ticonderoga', 'US|ab|ticonderoga_ny', 'US|ab|wayne_avenue', 'US|aw|85', 'US|aw|avenue', 'US|aw|ny', 'US|aw|ticonderoga']


,match,name,address,shared keys
0,S2-681193310,maure wilblims colombier inc,,"[US|nb|colombier_inc, US|nw|colombier, US|nw|inc, US|nw|maure]"
1,S2-743505751,maure williams colombier,,"[US|nb|maure_williams, US|nb|williams_colombier, US|nj|maurewillia..."
2,S3-860443364,maure williams inc center,,"[US|nb|maure_williams, US|nw|inc, US|nw|maure, US|nw|williams]"
3,S3-775321672,drexkor,85 wanye avenue ticonderoga townshiip new york,"[US|ab|avenue_ticonderoga, US|aw|85, US|aw|avenue, US|aw|ticonderoga]"
4,S3-11291185,maurewilliamscolombier,wayne avenue ticonderoga townshiip new york,"[US|ab|avenue_ticonderoga, US|ab|wayne_avenue, US|aw|avenue, US|aw..."


Every true match shares keys, but **different key types rescue different kinds of noise**:
- the two S2 records have **no address**, so only name keys (`maure`, `colombier`, `maure_williams`) can find them;
- `drexkor` has a completely different name and a typo (`wanye`), yet shares the address pair `avenue_ticonderoga`;
- the website-style name `maurewilliamscolombier` shares the **joined-name** key `nj|maurewilliamscolombier`
  — no single word or word pair would have linked it.

Common keys such as `nw|inc` or `aw|avenue` are shared too, but they appear in far more than 1,000 records,
so they are ignored for blocking.

---
## 3. Choosing the design

`experiments/phase7_blocking.py` built the key index **once** and compared 24 designs:

- `max_df` 100 / 300 / 1000 — the most common a key may be and still create candidates;
- ranking `all` (top K by idf sum over all shared keys) vs `split` (top K/2 by name keys ∪ top K/2 by address keys);
- K = 25 / 50 / 100 / 200 candidates per entity.

We choose on the **tuning** entities; the validation columns are shown only to confirm.

In [4]:
blocking = pd.read_csv(EXP / "phase7_blocking_results.tsv", sep="\t")
blocking.pivot_table(index=["max_df", "ranking"], columns="K", values="tuning_recall")

K                  25      50      100     200
max_df ranking                                
100    all      0.9014  0.9219  0.9363  0.9423
       split    0.8809  0.9117  0.9312  0.9416
300    all      0.9140  0.9296  0.9466  0.9598
       split    0.8934  0.9208  0.9409  0.9564
1000   all      0.9266  0.9392  0.9516  0.9623
       split    0.9027  0.9278  0.9449  0.9594

In [5]:
blocking[(blocking["max_df"] == 1000) & (blocking["ranking"] == "all")]

,max_df,ranking,K,tuning_recall,tuning_mean_candidates,tuning_no_candidates,validation_recall,validation_mean_candidates,validation_no_candidates
16,1000,all,25,0.9266,25.0,0,0.9277,25.0,0
17,1000,all,50,0.9392,49.9,0,0.9396,50.0,0
18,1000,all,100,0.9516,99.7,0,0.9522,99.8,0
19,1000,all,200,0.9623,198.4,0,0.9630,198.6,0


**Decision: `max_df = 1000`, ranking `all`, K = 50.**

- Candidate recall **0.58 → 0.94** (tuning 0.939, validation 0.940), with ~50 candidates per entity.
- `all` beats `split` for every K: letting name and address evidence add up ranks better than treating them separately.
- `max_df = 1000` leaves **no entity without candidates** (the baseline left 12% empty).
- K = 100 would add only +1.2 points of recall for twice the pairs (test set: 173M instead of 87M pairs to score).
  We can revisit this in Phase 12 if the model turns out to need it.

Reduction ratio on the test set: ~50 candidates out of ~2.9 million same-country records per entity ≈ **99.998%**.

---
## 4. End to end: new blocking + the Phase 6 rule scorer

In [6]:
pd.read_csv(EXP / "phase7_end_to_end.tsv", sep="\t").T

,0
blocking,"5 key types, max_df=1000, ranking=all, K=50"
scorer,Jaccard mean (Phase 6 rule)
tuning_candidate_recall,0.9392
validation_candidate_recall,0.9396
threshold,0.6
tuning_F0.5,0.7074
validation_F0.5,0.7058
validation_precision,0.7891
validation_recall,0.6704
candidate_pairs,1997854


**Same scorer, better candidates: validation F0.5 0.538 → 0.706 (+0.168).**

The scorer did not change at all — it simply gets to see 94% of the true matches instead of 59%.
This is why the problem statement says blocking "determines the upper bound of your recall".

---
# Summary — Phase 7

| | Phase 6 | Phase 7 |
|---|---|---|
| Keys | single words | words + word pairs + joined name |
| Candidate recall (validation) | 0.590 | **0.940** |
| Entities without candidates | ~12% | **0** |
| Candidates per entity | ~48 | ~50 |
| Validation F0.5 (Jaccard rule) | 0.538 | **0.706** |

**Lessons**
1. Diagnose before fixing: the baseline had the right *kind* of evidence but lost matches in **ranking**.
2. Word **pairs** are rare even when each word is common (`85_wayne`) — cheap, precise keys.
3. Different key types rescue different noise: name keys for empty addresses, address keys for different / foreign-script names,
   joined names for website names.
4. Integer codes + chunking + batches keep 10M records × 40k queries under ~2 GB of RAM.

**What's left:** 6% of true matches are still not candidates, and the scorer is crude (word Jaccard, one threshold).
Phase 8 builds richer similarity features on these candidates.